IMPORT LIBRARY

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import TimeSeriesSplit
import xgboost as xgb
from xgboost import XGBRegressor
from xgboost import plot_importance
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import math
import streamlit as st
import joblib


ModuleNotFoundError: No module named 'numpy'

DATA UNDERSTANDING

In [ ]:
# Load data dari file Excel
df = pd.read_excel('../data/raw/Order.all.20260101_20260131 (1).xlsx')

print("=" * 80)
print("1. DIMENSI DATA")
print("=" * 80)
print(f"Jumlah Baris: {df.shape[0]}")
print(f"Jumlah Kolom: {df.shape[1]}")
print(f"\nNama-Nama Kolom:\n{df.columns.tolist()}")

print("\n" + "=" * 80)
print("2. TIPE DATA")
print("=" * 80)
print(df.dtypes)

print("\n" + "=" * 80)
print("3. PREVIEW DATA (5 BARIS PERTAMA)")
print("=" * 80)
print(df.head())

print("\n" + "=" * 80)
print("4. STATISTIK DESKRIPTIF")
print("=" * 80)
print(df.describe())

print("\n" + "=" * 80)
print("5. MISSING VALUES")
print("=" * 80)
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({
    'Kolom': missing.index,
    'Jumlah Missing': missing.values,
    'Persentase (%)': missing_pct.values
})
print(missing_df)

print("\n" + "=" * 80)
print("6. DUPLIKAT DATA")
print("=" * 80)
print(f"Jumlah baris duplikat: {df.duplicated().sum()}")

print("\n" + "=" * 80)
print("7. INFORMASI LENGKAP")
print("=" * 80)
df.info()

In [ ]:
print("\n" + "=" * 80)
print("8. DISTRIBUSI KOLOM NUMERIK")
print("=" * 80)

# Ambil kolom numerik saja
numeric_cols = df.select_dtypes(include=[np.number]).columns
print(f"Kolom Numerik: {numeric_cols.tolist()}")

# Visualisasi distribusi
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.ravel()

for idx, col in enumerate(numeric_cols[:4]):
    axes[idx].hist(df[col], bins=30, edgecolor='black', color='skyblue')
    axes[idx].set_title(f'Distribusi {col}')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Frekuensi')
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "=" * 80)
print("9. ANALISIS KOLOM KATEGORI")
print("=" * 80)

categorical_cols = df.select_dtypes(include=['object']).columns
print(f"Kolom Kategori: {categorical_cols.tolist()}\n")

for col in categorical_cols[:5]:  # Tampilkan 5 kolom kategori pertama
    print(f"\n{col}:")
    print(df[col].value_counts())
    
print("\n" + "=" * 80)
print("10. OUTLIER DETECTION (NUMERICAL COLUMNS)")
print("=" * 80)

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    print(f"\n{col}: {len(outliers)} outlier(s) terdeteksi")